# Week 29, Day 3: The membrane as a capacitor: size, input resistance and recording

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w029/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Size, resistance and capacitance

For spherical cells of diameter 10, 20 and 40 μm, compute the capacitance `C` (farads), input resistance `R_in` (ohms), time constant `tau` (seconds) and the steady current `I_10` (amperes) that changes the voltage by 10 mV. Diameters are in μm; convert to cm.

In [ ]:
import numpy as np
c_m, r_m = 1e-6, 2e4                  # F/cm^2, Ohm cm^2
d = np.array([10.0, 20.0, 40.0])      # um
A = np.pi * (d * 1e-4) ** 2           # cm^2
C = ...
R_in = ...
tau = ...
I_10 = ...
for di, c, rr, t, i in zip(d, C, R_in, tau, I_10):
    print(f"d = {di:4.0f} um: C = {c * 1e12:6.1f} pF, R_in = {rr / 1e6:7.0f} MOhm, tau = {t * 1e3:.0f} ms, 10 mV needs {i * 1e12:5.1f} pA")

In [ ]:
import numpy as np
def test_values():
    "C, R_in and tau follow from the area pi d^2 with c_m = 1 uF/cm^2 and r_m = 20 kOhm cm^2"
    A = np.pi * (d * 1e-4) ** 2
    assert np.allclose(C, 1e-6 * A) and np.allclose(R_in, 2e4 / A) and np.allclose(tau, R_in * C)

def test_current():
    "I_10 is the current giving a 10 mV steady change, 0.01 / R_in"
    assert np.allclose(I_10, 0.01 / R_in) and np.allclose(tau, 0.02)

test_values()
test_current()
print("All checks passed")

### Exercise: Reading a cell from a voltage step

The cell below simulates the membrane test: a 10 mV step through Rs = 10 MΩ into a cell with R_in = 200 MΩ and C = 100 pF, with 5 pA of recording noise. Estimate the series resistance from the first sample, the input resistance from the steady current (the mean of the last 500 samples), and the capacitance from the transient charge Q, the integral of the current above its steady value, using C = (Q / ΔV) ((Rs + Rin) / Rin)².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(3)
Rs, Rin, Cm, dV, dt, n = 10e6, 200e6, 100e-12, 0.01, 1e-5, 2000
I = np.zeros(n); Vm = 0.0
for k in range(n):                       # Euler integration of the cell's voltage
    I[k] = (dV - Vm) / Rs
    Vm += dt * ((dV - Vm) / Rs - Vm / Rin) / Cm
I = I + r.normal(0, 5e-12, n)

Rs_est = ...
I_ss = I[-500:].mean()
Rin_est = ...
Q = ...
C_est = ...
print(f"Rs = {Rs_est / 1e6:.1f} MOhm, Rin = {Rin_est / 1e6:.0f} MOhm, C = {C_est * 1e12:.1f} pF")
print(f"transient time constant {1e3 * C_est * Rs_est * Rin_est / (Rs_est + Rin_est):.2f} ms")
plt.plot(np.arange(n) * dt * 1e3, I * 1e12); plt.xlabel("time (ms)"); plt.ylabel("current (pA)"); plt.ylim(-50, 300); plt.show()

In [ ]:
import numpy as np
def test_estimates():
    "the series resistance, input resistance and capacitance are each recovered within 3%"
    assert abs(Rs_est - 10e6) / 10e6 < 0.03 and abs(Rin_est - 200e6) / 200e6 < 0.03 and abs(C_est - 100e-12) / 100e-12 < 0.03

def test_method():
    "C_est uses the transient charge Q with the correction ((Rs + Rin) / Rin)^2"
    assert abs(C_est - Q / dV * ((Rs_est + Rin_est) / Rin_est) ** 2) < 1e-18

test_estimates()
test_method()
print("All checks passed")

### Exercise: Series resistance errors

For series resistances of 2, 5, 10 and 20 MΩ, with a fraction `comp` compensated by the amplifier (0 here), compute the voltage error `v_err` (mV) during a 5 nA sodium current and the clamp's time constant `tau_clamp` (ms) for a 100 pF cell, which is approximately the uncompensated Rs times C. Then rerun with 80% compensation.

In [ ]:
import numpy as np
Rs = np.array([2e6, 5e6, 10e6, 20e6])
comp = 0.0                                   # fraction of Rs compensated
v_err = ...
tau_clamp = ...
for r_, e_, t_ in zip(Rs, v_err, tau_clamp):
    print(f"Rs = {r_ / 1e6:4.0f} MOhm: error {e_:5.1f} mV, clamp time constant {t_:.2f} ms")

In [ ]:
import numpy as np
def test_error():
    "v_err is I times the uncompensated series resistance, in mV"
    assert np.allclose(v_err, 5e-9 * Rs * (1 - comp) * 1e3)

def test_speed():
    "tau_clamp is the uncompensated series resistance times C, in ms, and exceeds 0.5 ms for Rs = 10 MOhm without compensation"
    assert np.allclose(tau_clamp, Rs * (1 - comp) * 100e-12 * 1e3) and tau_clamp[2] > 0.5

test_error()
test_speed()
print("All checks passed")